# Customer Shopping Behavior - Data cleaning

**Obiettivo:** pulire e preparare il dataset di comportamento d'acquisto clienti per l'analisi.

**Dataset:** customer_shopping_behavior.csv

**Contenuto:** dati anagrafici clienti, prodotti acquistati, importi, rating, frequenza d'acquisto.

In [2]:
import pandas as pd
df = pd.read_csv("customer_shopping_behavior.csv")

In [3]:
#inizio a esplorare i dati
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

In [4]:
df.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [5]:
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [8]:
#ci sono 37 valori mancanti nella colonna "Review Rating"
df["Review Rating"] = df.groupby("Category")["Review Rating"].transform(lambda x: x.fillna(x.median()))
#Raggruppo i valori della colonna, per poi riempire i valori mancanti con la mediana del gruppo di appartenenza.
#In questo modo evito di usare un'unica mediana per l'intero dataset, mantenendo le differenze tra categorie.

In [10]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(" ","_")
df = df.rename(columns={"purchase_amount_(usd)":"purchase_amount"})

In [11]:
df.columns #controllo che le colonne siano standardizzate e pulite.

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

In [13]:
#Divido l'età in quartili invece di fasce fisse, così ogni gruppo contiene circa lo stesso numero di clienti, 
#rendendo i confronti tra fasce più bilanciati
labels = ["Young Adult", "Adult", "Middle-aged", "Senior"]
df["age_group"] = pd.qcut(df["age"], q=4, labels = labels)

In [14]:
df[["age", "age_group"]].head(10) #controllo il risultato

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


In [15]:
frequency_mapping = {"Fortnightly":14, 
                     "Weekly":7, 
                     "Monthly":30, 
                     "Quarterly":90, 
                     "Bi-Weekly":14, 
                     "Annually":365, 
                     "Every 3 Months":90}

df["purchase_frequency_days"] = df["frequency_of_purchases"].map(frequency_mapping)
#trasformo numericamente la frequenza d'acquisto tramite una mappatura

In [16]:
df[["purchase_frequency_days", "frequency_of_purchases"]].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [17]:
df[["discount_applied", "promo_code_used"]].head(10)
#le due colonne sembrano ridondanti

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [20]:
(df["discount_applied"] == df["promo_code_used"]).all()
#Faccio una verifica

np.True_

In [21]:
df = df.drop(columns=["promo_code_used"])
#Elimino la colonna ridondante, dopo averne avuto verifica

In [22]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='str')

In [23]:
df.to_csv("(cleaned)customer_shopping_behavior.csv", index=False)
#ottengo il file pulito

In [24]:
#Creo un databse SQLite e carico i dati come tabella "customer"
import sqlite3
conn = sqlite3.connect("shopping.db")
df.to_sql("customer", conn, if_exists="replace", index=False)
conn.close()
#Il lavoro continuerà su DB Browser utilizzando SQL per interrogare i dati e rispondere a domande di business.